# 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
GDRIVE_PATH = '/content/drive/MyDrive/DLPost'
print('GDrive connected:', os.path.exists(GDRIVE_PATH))

# 2. Tải Dataset Kaggle
Dataset ảnh AI vs Real images luôn được tải về làm dữ liệu huấn luyện mặc định.

In [ ]:
!pip install kagglehub -q
import kagglehub, os

kaggle_path = kagglehub.dataset_download('cashbowman/ai-generated-images-vs-real-images')
os.environ['KAGGLE_DATASET_DIR'] = kaggle_path
print('\n\u2705 Kaggle dataset path:', kaggle_path)

# 3. Clone / Cập nhật Repo

In [ ]:
import os
REPO_URL = 'https://github.com/PNCanh/DLPost.git'
REPO_DIR = '/content/DLPost'

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull origin main
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
print('\u2705 Repo ready')

# 4. Cài đặt thư viện

In [ ]:
%cd /content/DLPost
!pip install -r requirements.txt -q
print('\u2705 Dependencies installed')

# 5. Khởi tạo đường dẫn & thư mục

In [ ]:
import sys
sys.path.insert(0, '/content/DLPost/src')

from configs import paths
from configs.paths import ensure_directories
ensure_directories()

print(f'\u2705 IS_COLAB={paths.IS_COLAB}')
print(f'   ROOT_DIR     = {paths.ROOT_DIR}')
print(f'   DATASET_DIR  = {paths.DATASET_DIR}')
print(f'   OUTPUT_DIR   = {paths.OUTPUT_DIR}')

# 6. Kiểm tra GPU

In [ ]:
import torch
if torch.cuda.is_available():
    print(f'\u2705 GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('\u26a0\ufe0f  Không có GPU – vào Runtime > Change runtime type > GPU')

# 7. Xem cấu hình mô hình
Các cấu hình có sẵn được định nghĩa trong `src/configs/model_config.py`.

In [ ]:
from configs.model_config import CONFIGS

print('Cấu hình có sẵn:')
for key, cfg in CONFIGS.items():
    t = cfg['training']
    print(f"  {key:12s}  {cfg['text_model']}+{cfg['image_model']}  "
          f"epochs={t['epochs']}  lr={t['learning_rate']}  adv={t['adversarial']}")

# 8. Huấn luyện

Chọn **một trong hai** cell bên dưới:

| Option | Khi nào dùng |
|--------|-------------|
| **A – Full Pipeline** | Dữ liệu mới hoặc có thay đổi (OCR → Tiền xử lý → Split → Train → Evaluate) |
| **B – Train Only** | Dữ liệu đã được xử lý sẵn trên GDrive, chỉ train & evaluate model |

Thay đổi `SELECTED_CONFIG` để chọn mô hình (`baseline`, `variant_a`, `variant_b`).

Sau khi train xong, kết quả đánh giá (metrics, confusion matrix, predictions) sẽ được tự động lưu vào `outputs/`.

In [ ]:
# ====== Option A: FULL PIPELINE ======
SELECTED_CONFIG = 'baseline'  # baseline | variant_a | variant_b

!python /content/DLPost/src/main.py --config {SELECTED_CONFIG}

In [ ]:
# ====== Option B: TRAIN ONLY (dữ liệu đã xử lý sẵn) ======
SELECTED_CONFIG = 'baseline'  # baseline | variant_a | variant_b

!python /content/DLPost/src/main.py --config {SELECTED_CONFIG} --train_only

# 9. Xem kết quả đánh giá
Hiển thị metrics, classification report và confusion matrix đã được lưu sau khi train.

## 9.1 Metrics tổng hợp

In [ ]:
import json, glob
from configs import paths

# Tìm file report mới nhất
report_files = sorted(paths.METRICS_DIR.glob('*_report.json'))
if report_files:
    latest = report_files[-1]
    print(f'Đang đọc: {latest.name}\n')
    with open(latest, 'r', encoding='utf-8') as f:
        report = json.load(f)
    
    print('Model:', report.get('model_name', 'N/A'))
    print('Run:  ', report.get('run_name', 'N/A'))
    print()
    
    eval_metrics = report.get('evaluation_metrics', {})
    print('=== Evaluation Metrics ===')
    for k, v in eval_metrics.items():
        print(f'  {k:12s}: {v:.4f}')
else:
    print('Chưa có file report. Hãy chạy training trước.')

## 9.2 Classification Report

In [ ]:
if report_files:
    class_report = report.get('classification_report', {})
    if isinstance(class_report, dict):
        import pandas as pd
        # Nếu là dict dạng sklearn classification_report
        df_report = pd.DataFrame(class_report).transpose()
        display(df_report)
    elif isinstance(class_report, str):
        print(class_report)
    else:
        print(class_report)
else:
    print('Chưa có report.')

## 9.3 Metrics CSV

In [ ]:
import pandas as pd
from configs import paths

csv_files = sorted(paths.METRICS_DIR.glob('*_metrics.csv'))
if csv_files:
    latest_csv = csv_files[-1]
    print(f'File: {latest_csv.name}')
    df_metrics = pd.read_csv(latest_csv)
    display(df_metrics)
else:
    print('Chưa có metrics CSV.')

## 9.4 Confusion Matrix

In [ ]:
from IPython.display import Image, display
from configs import paths

cm_files = sorted(paths.FIGURES_DIR.glob('*_cm.png'))
if cm_files:
    latest_cm = cm_files[-1]
    print(f'File: {latest_cm.name}')
    display(Image(filename=str(latest_cm)))
else:
    print('Chưa có confusion matrix.')

## 9.5 Mẫu Predictions

In [ ]:
import json
from configs import paths

pred_files = sorted(paths.PREDICTIONS_DIR.glob('*_predictions.json'))
if pred_files:
    latest_pred = pred_files[-1]
    print(f'File: {latest_pred.name}\n')
    with open(latest_pred, 'r', encoding='utf-8') as f:
        preds = json.load(f)
    print(f'Tổng số predictions: {len(preds)}')
    print('\n--- 10 mẫu đầu tiên ---')
    for p in preds[:10]:
        print(f"  post={p.get('post_id','?'):12s}  "
              f"true={p['true_label']}  pred={p['predicted_label']}  "
              f"scam={p.get('is_scam', '?')}")
else:
    print('Chưa có predictions.')